# Singular AI Model approach

This approach focuses on a single LLM which is multi lingual to use the ` 58 reports ` and give a JSON file as an output which will be validated and saved if the labels are correct or goes back to training and labelling

In [2]:
import asyncio
import ollama
from ollama import chat
from pathlib import Path
import pandas as pd 

print("Import Successful")

Import Successful


In [32]:
# Make the prompt for LLM

prompt = """ 
You are extracting structured findings from a KNEE MRI radiology report..

Rules: 
1. Only use information that is explicitly present and stated in Report 
2. Do not infer information that are not written in the report 
3. Do not add any new information outside of the 12 labels that are present which are:
    'ACL', 'MCL', 'Medial Meniscus',
    'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', 'Baker's', 'Contusion', 'Fracture'
4. For each finding return one of: 
    - present : When there is an abnormality 
    - absent : When there is no abnormality 
    - uncertain : When abnormality is uncertain 
    - not_mentioned : When abnormality is not mentioned
5. Return valid JSON only 
6. For every answer except not_mentioned, copy a short evidence phrase from the report 


Report: 
---
{report}
---

Return JSON in this structure 
{{
    "ACL" : {{
            "status" : "present | absent | uncertain | not_mentioned"
            "evidence" : "exact phrase from the report"  
            }},
    "MCL" : {{
            "status" : "present | absent | uncertain | not_mentioned"
            "evidence" : "exact phrase from the report"  
            }}
}}

Do this for each of the 12 labels
'ACL', 'MCL', 'Medial Meniscus',
'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
'Synovitis', 'Baker's', 'Contusion', 'Fracture'
"""

# Prepare Report List

In [33]:
IN_DIR = Path("Sample_Data")
raw_data = IN_DIR / "Sample_Data.csv"
print("Data Import Successful")

# Isolate Report 
raw_df = pd.read_csv(raw_data)
reports = raw_df["Report"]

if all(reports):
    print("Report Isolation Successful")
else:
    print("Report Isolation Unsuccessful")

Data Import Successful
Report Isolation Successful


# JSON Creation

In [34]:
import json 
from tqdm import tqdm

results = []

for idx, report in tqdm(
    enumerate(reports[:10]),
    total=len(reports[:10])
):
    response = chat(
        model="gemma4:e4b",
        messages=[
            {
                "role": "user",
                "content": prompt.format(report=report)
            }
        ],
        format="json"
    )

    parsed = json.loads(response["message"]["content"])
    results.append({
        "idx": idx,
        "report": report,
        "model": "gemma3:4b",
        "output": parsed
    })

print("Complete")

100%|██████████| 10/10 [05:30<00:00, 33.09s/it]

Complete


In [35]:
OUT_DIR = Path("Sample_Data")
OUT_DIR.mkdir(exist_ok=True)

gemma_responses = pd.json_normalize(results)

gemma_responses = gemma_responses.drop(
    columns=["idx", "model"],
    errors="ignore"
)

gemma_responses.to_json(
    OUT_DIR / "Gemma-Responses.json",
    orient="records",
    indent=2,
    force_ascii=False
)